# ShadowMind — AI Detection Engine Training & Evaluation
### CNN (Spatial) + LSTM (Temporal) + MLP (Fusion Ensemble)

This notebook trains and evaluates the core detection engine on balanced network traffic flows and generates classification metrics (Accuracy, Precision, Recall, F1-Score, Confusion Matrix).

In [1]:
import sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# Ensure project root is on path
ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.preprocessor import load_and_prepare
from src.detection.engine import DetectionEngine

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.14.1+cpu
CUDA available: False


## 1. Load Data & Prepare Features

In [2]:
data = load_and_prepare()

X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]
X_train_seq, X_test_seq = data["X_train_seq"], data["X_test_seq"]
y_train_seq, y_test_seq = data["y_train_seq"], data["y_test_seq"]

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")
print(f"Test labels distribution:", np.bincount(y_test))

Train samples: 19204, Test samples: 4802
Test labels distribution: [2371 2431]


## 2. Initialize & Load Trained Detection Engine

In [3]:
engine = DetectionEngine()
engine.load_weights()
print("Engine loaded on device:", engine.device)

Engine loaded on device: cpu


## 3. Generate Evaluation Metrics on Test Set

In [4]:
engine.set_eval_mode()

# Align sizes
min_test = min(len(X_test), len(X_test_seq), len(y_test))
t_flat = torch.tensor(X_test[:min_test], dtype=torch.float32).to(engine.device)
t_seq  = torch.tensor(X_test_seq[:min_test], dtype=torch.float32).to(engine.device)
t_y    = y_test[:min_test]

with torch.no_grad():
    c_emb = engine.cnn(t_flat)
    l_emb = engine.lstm(t_seq)
    preds_tensor = engine.mlp(c_emb, l_emb)

scores = preds_tensor.cpu().numpy().flatten()
pred_labels = (scores >= 0.5).astype(int)

# Calculate metrics
acc = accuracy_score(t_y, pred_labels)
precision, recall, f1, _ = precision_recall_fscore_support(t_y, pred_labels, average='binary', zero_division=0)
cm = confusion_matrix(t_y, pred_labels)

print("=== SHADOWMIND DETECTION ENGINE PERFORMANCE ===")
print(f"Accuracy:  {acc * 100:.2f}%")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("\nConfusion Matrix:")
print(cm)

=== SHADOWMIND DETECTION ENGINE PERFORMANCE ===
Accuracy:  99.96%
Precision: 0.9992
Recall:    1.0000
F1-Score:  0.9996

Confusion Matrix:
[[2369    2]
 [   0 2381]]
